# Data merge and feature engineering


In [1]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Uw11\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [3]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [6]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
223280,2023-03-18 16:00:00,10,50.45060,30.52430,4.0,0.0,75.9,0.000,0.0,0.00,0.0,0.0,31.3,14.4,66.9,1027.0,78.5,126.9,10.9,4.0,0.88,6.9,4.8,63.73,0.0,0.0,0.0,23.8,10.4,91.3,1027.0,80.4,269.7,1.0,3.0,False,True,False,False,False,True,False,False,2023,3,5,16,Kyiv,Europe/Kiev,2023-03-18,06:06:02,18:07:03,16:00:00,Київська,0,0.000000
181488,2023-01-05 03:00:00,2,49.23360,28.44860,4.0,1.5,83.8,2.600,100.0,8.33,0.0,0.0,41.8,22.7,241.7,1009.2,95.8,6.0,0.5,0.0,0.42,0.6,-3.7,85.19,0.0,0.0,0.0,28.4,15.1,196.9,1016.0,97.3,0.0,0.1,0.0,False,False,True,False,False,True,False,False,2023,1,3,3,Vinnytsia,Europe/Kiev,2023-01-05,08:00:55,16:22:18,03:00:00,Вінницька,0,0.000000
498021,2024-07-07 17:00:00,24,48.29240,25.93550,23.8,15.0,59.7,0.000,0.0,0.00,0.0,0.0,37.4,24.5,134.5,1015.8,32.2,307.9,26.6,9.0,0.05,29.9,29.7,41.19,0.0,0.0,0.0,32.0,15.1,146.0,1014.7,10.0,589.8,2.1,6.0,False,True,False,False,True,False,False,False,2024,7,6,17,Chernivtsi,Europe/Kiev,2024-07-07,05:24:08,21:18:05,17:00:00,Чернівецька,0,0.000000
234206,2023-04-06 16:00:00,17,50.61930,26.25130,4.5,4.0,96.6,11.943,100.0,8.33,0.3,1.1,41.8,25.1,24.1,1011.7,99.6,33.1,2.7,1.0,0.50,7.5,4.4,93.38,0.0,0.0,0.0,30.6,17.6,55.2,1010.0,100.0,63.1,0.2,1.0,False,False,False,True,False,True,False,False,2023,4,3,16,Rivne,Europe/Kiev,2023-04-06,06:41:07,19:54:58,16:00:00,Рівненська,0,0.000000
452402,2024-04-19 13:00:00,4,48.47530,35.01600,10.8,4.0,64.1,0.200,100.0,8.33,0.0,0.0,55.4,32.4,241.2,1007.6,64.9,185.7,16.0,5.0,0.35,13.3,13.3,48.97,0.1,100.0,0.0,51.5,25.2,249.3,1010.0,56.9,524.9,1.9,5.0,False,False,True,False,False,False,True,False,2024,4,4,13,Dnipro,Europe/Kiev,2024-04-19,05:41:31,19:37:29,13:00:00,Дніпропетровська,1,16.333333
604781,2025-01-09 02:00:00,7,48.62640,22.28510,6.4,4.4,87.5,1.996,100.0,66.67,0.0,2.1,41.0,24.1,187.3,1010.5,92.5,6.2,0.6,0.0,0.34,4.8,2.2,89.29,0.0,0.0,0.0,36.0,11.0,211.0,1011.8,89.3,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2025,1,3,2,Uzhgorod,Europe/Uzhgorod,2025-01-09,08:21:35,16:54:56,02:00:00,Закарпатська,0,0.000000
323603,2023-09-08 21:00:00,14,46.97340,31.98520,18.9,4.4,41.9,0.000,0.0,0.00,0.0,0.0,31.7,18.0,24.6,1022.2,0.5,234.4,20.4,7.0,0.78,18.4,18.4,37.40,0.0,0.0,0.0,12.6,7.2,10.0,1019.8,0.0,0.0,0.0,0.0,True,False,False,False,True,False,False,False,2023,9,4,21,Mykolaiv,Europe/Kiev,2023-09-08,06:19:55,19:19:02,21:00:00,Миколаївська,0,0.000000
104917,2022-08-25 04:00:00,16,49.58790,34.55170,24.1,9.5,42.2,0.000,0.0,0.00,0.0,0.0,45.7,25.2,73.5,1017.7,39.0,259.9,22.4,8.0,0.92,17.9,17.9,59.49,0.0,0.0,0.0,16.9,8.3,33.9,1019.0,33.6,0.0,0.1,0.0,False,True,False,False,False,True,False,False,2022,8,3,4,Poltava,Europe/Kiev,2022-08-25,05:46:49,19:40:06,04:00:00,Полтавська,1,46.883333
141744,2022-10-28 03:00:00,2,49.23360,28.44860,10.1,7.8,86.3,0.000,0.0,0.00,0.0,0.0,20.9,9.4,237.1,1028.4,76.9,79.0,6.8,3.0,0.08,6.9,6.9,95.30,0.0,0.0,0.0,4.7,2.9,236.5,1030.0,28.1,0.0,0.1,0.0,False,True,False,False,False,True,False,False,2022,10,4,3,Vinnytsia,Europe/Kiev,2022-10-28,07:47:13,17:52:04,03:00:00,Вінницька,0,0.000000
188419,2023-01-17 03:00:00,22,49.41680,26.97430,3.3,2.0,88.9,0.100,100.0,4.17,0.0,0.0,46.8,36.0,162.8,999.5

In [7]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.5,91.76,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.4,89.80,0.0,0.0,0.0,16.6,8.3,289.2,1021.0,97.9,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,1,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,01:00:00,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.0,-0.3,90.44,0.0,0.0,0.0,27.7,7.6,287.6,1021.0,98.2,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,2,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,02:00:00,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,1.9,-0.3,91.75,0.0,0.0,0.0,15.1,7.2,288.9,1021.0,98.8,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,3,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,03:00:00,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,1.8,-0.2,92.40,0.0,0.0,0.0,13.7,6.5,290.4,1021.0,100.0,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,4,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,04:00:00,Вінницька,0,0.0


In [8]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 634680 entries, 0 to 634679
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  634680 non-null  datetime64[us]
 1   region_id                      634680 non-null  int64         
 2   city_latitude                  634680 non-null  float64       
 3   city_longitude                 634680 non-null  float64       
 4   day_temp                       634680 non-null  float64       
 5   day_dew                        634680 non-null  float64       
 6   day_humidity                   634680 non-null  float64       
 7   day_precip                     634680 non-null  float64       
 8   day_precipprob                 634680 non-null  float64       
 9   day_precipcover                634680 non-null  float64       
 10  day_snow                       634680 non-null  float64       
 11  day_snowdepth   

In [9]:
df_final = df_final.sort_values(["region_id", "datetime_hour"])

df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)

lag_cols = ["alarm_lag_1","alarm_lag_3","alarm_lag_6","alarm_lag_12"]
df_final[lag_cols] = df_final[lag_cols].fillna(0)

df_final['alarms_in_last_24h'] = (df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum()))

df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

hourly_total = df_final.groupby('datetime_hour')['alarm_active'].sum().shift(1)
df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)

df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1
196989,2023-02-01 00:00:00,24,48.2924,25.9355,1.6,-2.9,72.9,0.006,100.0,8.33,0.0,0.1,50.4,24.4,269.0,1011.2,75.3,62.4,5.4,3.0,0.35,2.0,-2.0,74.81,0.0,0.0,0.0,30.2,14.8,320.0,1012.0,50.0,0.0,0.0,0.0,False,False,False,True,False,True,False,False,2023,2,2,0,Chernivtsi,Europe/Kiev,2023-02-01,07:45:24,17:14:51,00:00:00,Чернівецька,0,0.000000,0.0,0.0,0.0,1.0,4.0,0,1,1.0
466343,2024-05-13 17:00:00,26,50.4506,30.5243,7.9,1.0,63.5,0.500,100.0,20.83,0.0,0.0,35.3,15.5,31.1,1020.7,68.8,206.9,18.0,6.0,0.17,10.4,10.4,51.02,0.0,0.0,0.0,31.0,14.0,39.2,1020.0,80.5,333.0,1.2,3.0,False,False,True,False,False,True,False,False,2024,5,0,17,Kyiv,Europe/Kiev,2024-05-13,05:12:46,20:36:48,17:00:00,Київ,1,9.233333,1.0,1.0,0.0,0.0,5.0,0,0,24.0
612023,2025-01-21 15:00:00,26,50.4506,30.5243,-0.6,-1.0,96.9,0.000,0.0,0.00,0.1,0.0,30.6,16.9,149.2,1025.8,98.8,22.8,1.9,1.0,0.75,-0.8,-5.1,98.55,0.0,0.0,0.0,26.3,13.3,147.9,1025.0,100.0,58.9,0.2,1.0,False,True,False,False,False,True,False,False,2025,1,1,15,Kyiv,Europe/Kiev,2025-01-21,07:46:47,16:32:19,15:00:00,Київ,0,0.000000,0.0,0.0,0.0,0.0,3.0,0,0,0.0
415410,2024-02-15 06:00:00,21,46.6401,32.6142,6.6,5.1,90.8,3.300,100.0,8.33,0.0,0.0,35.3,21.6,43.1,1019.0,99.7,53.0,4.6,2.0,0.19,7.5,5.1,98.64,0.0,0.0,0.0,22.7,13.0,36.4,1017.0,100.0,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2024,2,3,6,Kherson,Europe/Kiev,2024-02-15,06:54:33,17:13:36,06:00:00,Херсонська,1,60.000000,1.0,0.0,0.0,0.0,5.0,0,1,24.0
482613,2024-06-10 23:00:00,24,48.2924,25.9355,21.2,16.8,77.5,1.156,100.0,8.33,0.0,0.0,38.2,20.7,192.9,1007.4,63.2,248.3,21.4,8.0,0.12,19.2,19.2,86.50,0.0,0.0,0.0,20.9,20.7,256.0,1006.9,88.0,0.0,0.0,0.0,False,False,True,False,False,True,False,False,2024,6,0,23,Chernivtsi,Europe/Kiev,2024-06-10,05:15:37,21:16:25,23:00:00,Чернівецька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,1,3.0
11862,2022-03-16 14:00:00,8,47.8289,35.1626,-1.0,-7.3,63.5,0.000,0.0,0.00,0.0,0.0,31.3,16.2,13.5,1030.0,43.5,201.3,17.3,6.0,0.45,3.4,3.4,68.74,0.0,0.0,0.0,31.0,16.2,290.0,1027.8,40.0,592.0,2.1,6.0,False,True,False,False,False,True,False,False,2022,3,2,14,Zaporozhye,Europe/Zaporozhye,2022-03-16,05:50:53,17:46:01,14:00:00,Запорізька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,2.0
400293,2024-01-20 00:00:00,24,48.2924,25.9355,-2.1,-5.7,77.0,0.657,100.0,8.33,0.6,3.0,36.0,22.7,300.0,1024.7,37.3,57.2,4.8,3.0,0.31,-2.4,-7.2,95.74,0.0,0.0,0.6,21.6,14.7,315.0,1019.3,25.0,0.0,0.0,0.0,False,False,False,True,False,True,False,False,2024,1,5,0,Chernivtsi,Europe/Kiev,2024-01-20,07:58:59,16:55:46,00:00:00,Чернівецька,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,1,5.0
113841,2022-09-09 16:00:00,11,48.5085,32.2656,14.7,8.5,69.8,1.100,100.0,8.33,0.0,0.0,29.9,21.6,70.0,1020.6,78.9,148.8,12.9,7.0,0.42,20.1,20.1,42.02,0.0,0.0,0.0,25.9,11.9,78.6,1021.0,98.2,43.0,0.2,0.0,False,False,True,False,False,True,False,False,2022,9,4,16,Kropyvnytskyi,Europe/Kiev,2022-09-09,06:18:58,19:16:49,16:00:00,Кіровоградська,0,0.000000,0.0,0.0,0.0,0.0,3.0,0,0,0.0
518272,2024-08-11 21:00:00,19,49.5527,25.5889,21.4,15.7,72.4,0.800,100.0,4.17,0.0,0.0,45.7,21.6,285.0,1

In [10]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(
    index='datetime_hour',
    columns='region_id',
    values='alarm_active',
    fill_value=0
)

neighbour_alarm_matrix = pd.DataFrame(index=alarms_matrix.index)

for region, neighbours in neighbouring_regions.items():
    valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]
    
    if valid_neighbours:
        neighbour_alarm_matrix[region] = alarms_matrix[valid_neighbours].sum(axis=1)
    else:
        neighbour_alarm_matrix[region] = 0

neighbour_alarm_matrix = neighbour_alarm_matrix.shift(1)

neighbour_alarm_long = (neighbour_alarm_matrix.stack().reset_index())

neighbour_alarm_long.columns = ['datetime_hour','region_id','neighbour_alarms']

df_final = df_final.merge(neighbour_alarm_long,on=['datetime_hour', 'region_id'], how='left')

In [11]:
def hours_since_last_alarm(series):
    shifted = series.shift(1).fillna(0)
    result = []
    count = 0
    for val in shifted:
        if val == 1:
            count = 0
        else:
            count += 1
        result.append(count)
    return pd.Series(result, index=series.index)

df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(hours_since_last_alarm))

In [12]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
361627,2024-03-08 12:00:00,16,49.5879,34.5517,-0.9,-6.4,67.5,0.000,0.0,0.00,0.0,0.0,23.0,10.8,288.4,1021.3,56.3,88.8,7.6,3.0,0.93,1.7,-0.5,54.50,0.0,0.0,0.0,18.0,7.2,278.8,1022.0,85.9,283.9,1.0,3.0,False,True,False,False,False,True,False,False,2024,3,4,12,Poltava,Europe/Kiev,2024-03-08,06:09:32,17:36:20,12:00:00,Полтавська,0,0.000000,0.0,0.0,0.0,0.0,7.0,0,0,4.0,2.0,3
352600,2023-02-26 08:00:00,16,49.5879,34.5517,3.7,2.0,88.4,1.000,100.0,4.17,0.0,0.1,48.6,25.6,240.3,1008.3,99.0,56.2,4.8,3.0,0.20,2.9,0.9,91.16,0.0,0.0,0.0,14.0,7.2,250.0,1006.2,100.0,29.0,0.1,0.0,False,False,True,False,False,True,False,False,2023,2,6,8,Poltava,Europe/Kiev,2023-02-26,06:31:31,17:18:49,08:00:00,Полтавська,0,0.000000,0.0,0.0,0.0,0.0,3.0,1,0,0.0,0.0,15
148912,2024-01-20 09:00:00,7,48.6264,22.2851,-1.7,-6.6,70.2,0.001,100.0,4.17,0.0,6.4,33.8,10.5,260.7,1027.3,29.2,69.4,6.0,3.0,0.31,-5.8,-5.8,84.82,0.0,0.0,0.0,14.4,4.5,220.0,1027.4,23.5,15.0,0.1,0.0,False,False,False,True,False,True,False,False,2024,1,5,9,Uzhgorod,Europe/Uzhgorod,2024-01-20,08:14:45,17:09:13,09:00:00,Закарпатська,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,0,0.0,0.0,43
355128,2023-06-11 17:00:00,16,49.5879,34.5517,18.2,12.8,71.4,5.900,100.0,16.67,0.0,0.0,52.2,36.0,55.0,1012.3,95.5,201.3,17.5,7.0,0.76,17.0,17.0,88.58,2.1,100.0,0.0,43.6,19.4,57.1,1013.0,100.0,157.7,0.6,2.0,False,False,True,False,False,False,True,False,2023,6,6,17,Poltava,Europe/Kiev,2023-06-11,04:34:58,20:48:13,17:00:00,Полтавська,1,17.433333,0.0,0.0,0.0,0.0,7.0,1,0,2.0,1.0,1
395887,2025-01-28 04:00:00,17,50.6193,26.2513,5.2,3.0,85.9,0.000,0.0,0.00,0.0,0.0,34.9,19.4,163.6,1010.7,81.5,42.8,3.8,2.0,0.96,3.1,-0.8,93.14,0.0,0.0,0.0,30.2,15.8,171.4,1012.0,17.6,0.0,0.0,0.0,False,True,False,False,True,False,False,False,2025,1,1,4,Rivne,Europe/Kiev,2025-01-28,07:56:01,17:00:37,04:00:00,Рівненська,0,0.000000,0.0,0.0,0.0,0.0,1.0,0,1,12.0,0.0,22
597775,2023-12-22 03:00:00,25,51.4937,31.2944,1.3,0.0,91.3,2.000,100.0,4.17,0.2,1.3,43.9,24.1,185.8,986.2,93.4,10.8,0.9,1.0,0.34,1.2,-4.1,85.26,0.0,0.0,0.0,40.7,22.3,179.7,990.0,100.0,0.0,0.0,0.0,False,False,False,True,False,True,False,False,2023,12,4,3,Chernihiv,Europe/Kiev,2023-12-22,07:58:27,15:48:06,03:00:00,Чернігівська,0,0.000000,0.0,1.0,1.0,0.0,5.0,0,1,4.0,0.0,2
511601,2023-03-12 03:00:00,22,49.4168,26.9743,0.4,-3.9,74.4,1.000,100.0,4.17,1.0,1.1,59.0,36.0,291.9,1008.2,61.7,121.9,10.7,5.0,0.67,-0.5,-7.5,92.95,0.0,0.0,0.3,53.3,33.1,309.0,1001.0,100.0,0.0,0.0,0.0,False,False,False,True,False,True,False,False,2023,3,6,3,Khmelnytskyi,Europe/Kiev,2023-03-12,06:32:52,18:11:55,03:00:00,Хмельницька,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,1,0.0,0.0,31
153458,2024-07-27 20:00:00,7,48.6264,22.2851,21.1,12.3,61.1,0.008,100.0,4.17,0.0,0.0,25.9,12.5,189.1,1017.4,25.8,296.2,25.7,8.0,0.71,26.6,26.6,41.62,0.0,0.0,0.0,16.2,12.5,189.0,1016.6,28.4,182.6,0.7,2.0,False,False,True,False,False,True,False,False,2024,7,5,20,Uzhgorod,Europe/Uzhgorod,2024-07-27,05:58:56,21:15:09,20:00:00,Закарпатська,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,0,1.0,0.0,43


In [13]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
476,2023-06-17,0.771852,-0.185982,-0.087620,-0.064521,-0.093318,0.096884,0.025313,0.026478,-0.112319,0.113582,0.027761,0.161396,-0.082255,0.039890,0.008128,-0.085181,-0.038895,0.004820,0.033247,-0.009073,0.085051,0.010810,0.029528,-0.019612,0.029960,-0.062320,0.004967,-0.019809,-0.016240,-0.015038,-0.084560,-0.005676,0.052297,-0.036054,0.012347,0.004358,-0.045601,0.014039,-0.022453,0.018126,-0.036409,-0.037863,-0.066028,-0.093790,0.078159,-0.014084,0.017645,-0.056756,0.029922,-0.027201,0.021642,0.002797,0.055198,-0.024372,-0.030874,0.013387,0.010554,0.047862,0.014339,0.054929,0.002446,0.043620,-0.024838,0.015616,-0.041019,-0.042120,0.011651,-0.052155,-0.013579,0.034463,0.048394,-0.063770,0.043129,-0.009714,-0.020040,0.006054,-0.056869,-0.022954,-0.028101,0.000157,-0.003192,0.000913,-0.014853,-0.016374,0.050575,0.060122,0.009242,-0.030965,0.048860,-0.047804,-0.009138,0.024654,-0.016397,0.035929,-0.005852,0.000161,0.060528,-0.007489,-0.004289,0.013465,0.035507,0.022537,-0.008198,-0.013873,-0.024556,-0.038209,-0.073010,-0.006038,-0.007731,0.027309,-0.006509,-0.010957,0.032716,0.001310,0.004550,-0.029069,0.021539,0.051481,-0.046010,0.002111,0.009508,0.039940,0.024596,0.016278,-0.022953,0.023360,0.037625,0.000632,-0.032786,0.023138,-0.030634,-0.006287,-0.017008,-0.017396,0.009800,0.000658,0.002732,-0.021685,0.009420,0.000091,0.012596,0.000425,-0.014978,-0.015641,-0.013823,0.010228,-0.033134,-0.013788,-0.014273,0.014020
1247,2025-08-03,0.801591,0.282454,0.132537,-0.088477,-0.133454,0.093214,-0.017846,0.132965,0.110878,-0.050596,0.102088,-0.006794,-0.117475,-0.097799,-0.040836,-0.019058,0.012927,0.004710,-0.007879,0.012942,-0.062645,0.012944,0.029404,-0.012616,-0.060775,0.083123,-0.009078,0.034464,-0.032402,0.020468,-0.004073,-0.005226,-0.001602,0.043697,0.018273,-0.006623,0.016680,0.010126,-0.003353,0.012944,-0.005818,0.019297,0.019035,0.014342,0.012017,0.002447,-0.006419,0.004043,-0.017366,0.014667,-0.009565,0.002654,0.017363,0.029377,0.009296,-0.012470,-0.014540,-0.007841,0

In [14]:
#df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)    ТУТ ЗСУВ ІСВ НА + 1 ДЕНЬ
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [15]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [16]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [38]:
df_final.head(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,t

In [18]:
df_final.isna().sum()

datetime_hour        0
region_id            0
city_latitude        0
city_longitude       0
day_temp             0
                  ... 
isw_topic_145     5760
isw_topic_146     5760
isw_topic_147     5760
isw_topic_148     5760
isw_topic_149     5760
Length: 216, dtype: int64

In [19]:
df_final.fillna(0, inplace=True)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,0.0,0.0,27.7,10.8,312.0,1023.4,82.5,60.4,5.4,3.0,0.77,2.1,-0.5,91.76,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,False,False,True,False,True,False,False,2022,2,3,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,1,0.648336,-0.079572,0.178041,0.061854,0.073453,-0.055809,-0.011311,0.047310,0.008354,0.006474,-0.047099,0.020496,0.050190,-0.038573,0.057228,-0.072471,0.045393,-0.116783,0.168249,0.167355,-0.025072,-0.073858,-0.097024,-0.146191,0.027983,0.082989,0.014423,0.058370,-0.033928,0.046265,0.006901,-0.049498,0.126117,-0.089039,-0.006767,-0.031686,-0.010346,0.025637,0.061278,0.162580,-0.028081,0.161291,-0.018217,-0.002657,0.038485,-0.000530,0.044413,0.107176,-0.050721,0.064279,-0.066677,0.016742,-0.050257,0.072549,-0.035201,-0.027279,0.017115,0.043631,0.024660,-0.025841,0.031320,-0.076959,-0.072986,-0.040465,0.003958,-0.022475,0.033208,-0.028056,-0

In [20]:
df_final.isna().sum()

datetime_hour     0
region_id         0
city_latitude     0
city_longitude    0
day_temp          0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 216, dtype: int64

### Feature engineering for isw topics 

In [21]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final['isw_topic_mean'] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = (
    df_final.groupby('region_id')['isw_total_intensity']
    .diff(24)
    .fillna(0)
)

df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))

df_final['isw_topic_entropy'] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

isw_cols = ['isw_velocity_24h', 'isw_intensity_ema', 'isw_topic_entropy']
df_final[isw_cols] = df_final[isw_cols].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_9376\1269306822.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_9376\1269306822.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_9376\1269306822.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfor

In [22]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy
628096,2024-05-07 16:00:00,26,50.4506,30.5243,14.8,9.7,72.5,0.3,100.0,4.17,0.0,0.0,49.0,25.6,299.2,1008.5,79.8,180.7,15.5,6.0,0.97,18.8,18.8,54.00,0.0,0.0,0.0,46.1,23.8,307.8,1008.0,42.5,427.4,1.5,4.0,False,False,True,False,False,True,False,False,2024,5,1,16,Kyiv,Europe/Kiev,2024-05-07,05:22:06,20:27:52,16:00:00,Київ,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1.0,0.0,72,0.760383,-0.115209,-0.216867,0.116622,-0.026353,-0.001564,0.029514,-0.019226,-0.002838,-0.035358,-0.015497,0.022991,0.050022,-0.055749,-0.096895,0.146493,0.152837,0.008929,0.007954,0.006124,-0.047747,-0.032945,0.004861,0.020967,0.082041,-0.075820,-0.008549,0.010871,-0.034577,0.031983,0.001599,0.005368,0.000723,-0.031691,0.006319,0.016787,0.064792,0.045437,0.021521,-0.025317,0.012631,0.015906,-0.005648,-0.011209,0.012172,-0.028425,-0.015082,0.050468,-0.058386,0.015673,-0.013047,-0.043043,-0.000638,0.039013,0.027697,-0.052935,-

### TELEGRAM MERGE

In [23]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [24]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,0.042624,-0.024341,-0.018764,0.023404,-0.017137,-0.358836,0.425288,0.017966,-0.107660,-0.032066,-0.045986,0.118212,0.091172,-0.248637,-0.001069,0.150763,0.030603,-0.074030,0.005803,0.005871,0.034871,0.017239,0.058098,0.029814,-0.010845,0.031486,-0.022705,0.005607,-0.006324,-0.010169,0.024500,-0.015581,-0.012708,0.005139,0.007857,0.016734,-0.040289,-0.019688,-0.026763,0.009381,-0.043500,-0.009525,-0.013075,-0.010473,-0.018150,0.005988,0.032611,0.000850,0.017872,0.020885,0.023899,-0.019290,-0.006676,-0.018478,-0.023263,-0.005497,-0.006460,0.026051,0.004991,0.019687,-0.022244,0.010929,0.007016,-0.067349,-0.035249,-0.005686,0.018747,0.039159,-0.053868,-0.078189,-0.074737,-0.082163,0.020498,0.084336,0.113470,-0.025871,-0.048314,0.096

In [25]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

C:\Users\Uw11\AppData\Local\Temp\ipykernel_9376\2851151304.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")


In [26]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]

tg_hourly = (
    df_tg_matrix.groupby("datetime_hour")[topic_cols]
    .mean()
    .sort_index()
    .reset_index()
)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_9376\2188712141.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


In [27]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,0.042624,-0.024341,-0.018764,0.023404,-0.017137,-0.358836,0.425288,0.017966,-0.107660,-0.032066,-0.045986,0.118212,0.091172,-0.248637,-0.001069,0.150763,0.030603,-0.074030,0.005803,0.005871,0.034871,0.017239,0.058098,0.029814,-0.010845,0.031486,-0.022705,0.005607,-0.006324,-0.010169,0.024500,-0.015581,-0.012708,0.005139,0.007857,0.016734,-0.040289,-0.019688,-0.026763,0.009381,-0.043500,-0.009525,-0.013075,-0.010473,-0.018150,0.005988,0.032611,0.000850,0.017872,0.020885,0.023899,-0.019290,-0.006676,-0.018478,-0.023263,-0.005497,-0.006460,0.026051,0.004991,0.019687,-0.022244,0.010929,0.007016,-0.067349,-0.035249,-0.005686,0.018747,0.039159,-0.053868,-0.078189,-0.074737,-0.082163,0.020498,0.084336,0.113470,-0.025871,-

In [28]:
#tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)        ТУТ ЗСУВ ТГ НА 1 ГОДИНУ 

In [29]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [30]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,t

In [31]:
df_final.shape

(635256, 473)

In [32]:
df_final = df_final.sort_values(["datetime_hour"])

In [33]:
with pd.option_context('display.max_rows', None):
    print(df_final.isnull().sum())

datetime_hour                         0
region_id                             0
city_latitude                         0
city_longitude                        0
day_temp                              0
day_dew                               0
day_humidity                          0
day_precip                            0
day_precipprob                        0
day_precipcover                       0
day_snow                              0
day_snowdepth                         0
day_windgust                          0
day_windspeed                         0
day_winddir                           0
day_pressure                          0
day_cloudcover                        0
day_solarradiation                    0
day_solarenergy                       0
day_uvindex                           0
day_moonphase                         0
hour_temp                             0
hour_feelslike                        0
hour_humidity                         0
hour_precip                           0


In [34]:
df_final = df_final.fillna(0)

In [35]:
tg_cols = [c for c in df_final.columns if 'tg_topic_' in c]
df_tg_abs = df_final[tg_cols].abs()

df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
df_final['tg_topic_max'] = df_tg_abs.max(axis=1)
df_final['tg_topic_entropy'] = -(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) * np.log(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_9376\409095542.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_9376\409095542.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_9376\409095542.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance. 

In [36]:
df_final.head(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_snow,day_snowdepth,day_windgust,day_windspeed,day_winddir,day_pressure,day_cloudcover,day_solarradiation,day_solarenergy,day_uvindex,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_precip,hour_precipprob,hour_snow,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_conditions_simple_Clear,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,year,month,day_of_week,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,t

In [37]:
df_final.to_parquet("merged_dataset.parquet", index=False, engine="pyarrow")